In [ ]:
from option_analyzer import *
from indicators import compute_emas
self = OptionAnalyzer('quotes', 'chain')
pd.set_option("display.max_columns", None)

### Run this once every day to load close prices in the past 60 days

In [ ]:
df_close = pd.read_csv('output/yf_close.csv', parse_dates=['Date']).set_index('Date').tail(60)
df_close.columns.name = 'symbol'
latest_bollinger_file = max(glob('output/bollinger*.csv'))
df_boll = pd.read_csv(latest_bollinger_file).set_index('symbol')
df_boll = df_boll.loc[:, ['LB20', 'UB20', 'MA20', 'LB30', 'UB30', 'MA30']]
today = pd.Timestamp.now().normalize()
print('df_close data age:', today - df_close.index[-1])
print('latest Bollinger file:', latest_bollinger_file)

In [ ]:
os.system('sync > /dev/null 2>&1')
option_type = 'call'
servers = sorted(set([f.split('~')[1] for f in glob(os.path.expanduser(f'~/lab/data/{option_type}~*~*.csv'))]))
latest_option_files = [sorted(glob(os.path.expanduser(f'~/lab/data/{option_type}~{svr}~*.csv')))[-1] for svr in servers]
print('\n'.join(['%40s' % _ for _ in map(os.path.basename, latest_option_files)]))
chain_file_mtimes = dict([(os.path.basename(_f), os.path.getmtime(_f)) for _f in glob(os.path.expanduser('~/lab/chain/*'))])
latest_symbol = sorted(chain_file_mtimes, key=chain_file_mtimes.get)[-1]
print('Last symbol:', latest_symbol, datetime.fromtimestamp(chain_file_mtimes[latest_symbol]).strftime('%F %T'))
dfc = pd.concat([pd.read_csv(_f) for _f in latest_option_files])

df_today = dfc.loc[:, ['symbol', 'lastPrice']].drop_duplicates().rename(columns={'lastPrice': today}).set_index('symbol')
if df_today.columns[0] in df_close.T.columns:
    df_price = df_close
else:
    df_price = df_close.T.join(df_today, how='right').T
df_ema = compute_emas(df_price, [21, 50])

_df = dfc[(dfc.symbol=='QQQ') & (dfc.dthStrikeMargin >= 0)].sort_values(by='dthProfit', ascending=False).drop(columns=['overpaid', 'leverage'])
px.scatter(_df.head(100), x='dthStrikeMargin', y='dthProfit', color='expDt', height=400, width=1500).show()
_df.head(10)

In [ ]:
dfc[(dfc.symbol=='MRVL') & (dfc.strike==250) & (dfc.dte <= 45)].sort_values(by='dteProfit', ascending=False).head(10)

In [ ]:
dfq = dfc[(dfc.symbol=='MRVL') & (dfc.expDt=='2026-08-28') & (dfc.Delta <= 0.5)]
px.bar(dfq, x='strike', y='OpenInterest', color='Delta')

In [ ]:
_dfl = dfl.sort_values(by='OpenInterest', ascending=False).head(100)
px.scatter(_dfl, x='leverage', y='OpenInterest', width=1500, height=1000, color='option').show()
_dfl.head(10)

In [ ]:
(200+240)/419.6

### Leaps of different leverages

In [ ]:
dfl = dfc[(dfc.symbol=='GLD') & (dfc.dte >= 60) & (dfc.overpaid <= 0.03)].drop(columns=['dteProfit', 'dthProfit', 'dthStrikeMargin']).copy()
dfl['theta_pct'] = 100*dfl.Theta/dfl.mid
dfl['option'] = dfl.symbol + ' ' + dfl.expDt + ' ' + dfl.strike.astype(str)
_res = []
for leverage in np.arange(3, 12, 0.25):
    _dfl = dfl[(dfl.leverage >= leverage)]
    if _dfl.shape[0] == 0:
        continue
    _dfl = _dfl.sort_values(by='theta_pct', ascending=False).head(1)
    if len(_res) == 0 or _res[-1].index != _dfl.index:
        _res.append(_dfl)
px.scatter(pd.concat(_res), x='leverage', y='theta_pct', color='option', width=1000, height=600).show()
pd.concat(_res).sort_values(by='theta_pct', ascending=False)

In [ ]:
len(_res)

In [ ]:
# Leaps
dfl = dfc[(dfc.dte >= 180) & (dfc.symbol == 'GLD') & (dfc.Delta >= 0.5)].drop(columns=['dteProfit', 'dthProfit', 'dthStrikeMargin'])
dfl['theta_pct'] = 100*dfl.Theta/dfl.mid
dfl = dfl[dfl.leverage >= 3].sort_values(by='theta_pct', ascending=False)
px.scatter(dfl, x='strike', y='theta_pct', color='expDt', width=1000, height=600).show()
dfl[~dfl.symbol.str.contains('_')].head(20)

In [ ]:
# Sell calls
#dfc = dfc.drop(columns=['overpaid', 'leverage'])
_dfc = dfc[(dfc.dthStrikeMargin >= 10)].sort_values(by='dthProfit', ascending=False)
if _dfc.shape[0] > 0:
    _history_csv = pd.Timestamp.now().strftime('history/call~%F~%T.csv')
    _dfc.to_csv(_history_csv, index=None)
    print(_history_csv, os.path.getsize(_history_csv), 'bytes archived')
    print('Options after the filters:', _dfc.shape[0], 'out of', dfc.shape[0])
    px.scatter(_dfc.head(50), x='dthStrikeMargin', y='dthProfit', color='symbol', width=1200, height=600).show()
else:
    print('No good option.')

_dfc.head(20)

### Leaps: ignore no-bid or low open interest (minimum open interests is 100)
Buy calls to maximize leverage

In [ ]:
dte_lb = 90
hdte_resid_lb = 0.9
overpaid_ub = 0.1
spread_ub = 20
_filter = (dfc.dte >= dte_lb) & (dfc.leverage >= 3)
_filter = _filter & (dfc.hdte_resid >= hdte_resid_lb)
#_filter = _filter & (dfc.overpaid <= overpaid_ub) & (dfc.pctSpread <= spread_ub)
#_filter = _filter  & (dfc.moneyness <= 1.0) & (dfc.symbol != 'TLT') & (dfc.OpenInterest >= 1000)
_dfc = dfc[_filter].drop(columns=['dthr', 'dtzr']).sort_values(by='leverage', ascending=False).head(100)
print('Options after the filters:', _dfc.shape[0], 'out of', dfc.shape[0])
px.scatter(_dfc, x='hdte_resid', y='leverage', color='symbol', height=500).show()

_dfb = df_boll.join(df_today, how='right')
_dfb['rank20'] = (_dfb[today].astype(float) - _dfb['MA20'].astype(float))/(_dfb['UB20'] - _dfb['LB20'])*200
_dfb['rank30'] = (_dfb[today].astype(float) - _dfb['MA30'].astype(float))/(_dfb['UB30'] - _dfb['LB30'])*200
_dfb = _dfb.sort_values(by='rank20', ascending=False)
px.bar(_dfb, y=['rank20', 'rank30'], barmode='group').show()

df_last_ema = df_price.tail(1).T.join(df_ema.tail(1).T.unstack(level=1).droplevel(level=0, axis=1))
df_last_ema = df_last_ema.loc[list(_dfb.index)]
px.bar(df_last_ema.loc[:, reversed(df_last_ema.columns)], barmode='group').show()

_dfc.drop(columns=['dthProfit', 'dteProfit']).head(60)

In [ ]:
_df = dfc[(dfc.symbol=='MU') & (dfc.dte >= 90) & (dfc.overpaid <= 0.05)].sort_values(by='leverage', ascending=False).head(60)
px.scatter(_df, x='overpaid', y='leverage', color='expDt').show()
_df

### Top leverage

In [ ]:
_filter = (dfc.pctSpread <= 5) & (dfc.moneyness <= 1) & (dfc.dte >= 60) & (dfc.symbol != 'TLT')
_df = dfc[_filter].sort_values(by='leverage', ascending=False)
px.scatter(_df.head(200), x='hdte_resid', y='leverage', color='symbol', height=600).show()
_df.head(20)

In [ ]:
px.scatter(dfc[(dfc.symbol=='TSM') & (dfc.dte >= 90) & (dfc.hdte_resid >= 0.8)].sort_values(by='leverage', ascending=False).head(20), x='hdte_resid', y='mid', color='expDt', height=800)

In [ ]:
dfc[(dfc.symbol=='TSM')].dte.unique()# & (dfc.expDt >= '2026-07-17')]# & (dfc.hdte_resid >= 0.9)].drop(columns=['dteProfit', 'dthProfit', 'dthStrikeMargin', 'E'])

In [ ]:
px.scatter(dfc[dfc.symbol.str.contains('QQQ|SPY') & (dfc.dte >= 90) & (dfc.dte <= 300) & (dfc.moneyness <= 1) & (dfc.hdte_resid >= 0.9) & (dfc.hdte_resid <= 0.99)], x='hdte_resid', y='leverage', color='expDt', height=800)

### The End